# IAM LLM fine-tuning — **Qwen/Qwen3.5-9B** — Colab Pro / A100 **80GB**

Notes on the setup:

1. **`FastLanguageModel`, bf16, explicit `target_modules`.** Qwen3.5 ships as a
   unified VLM checkpoint (`Qwen3_5ForConditionalGeneration`; no text-only 9B), but
   Unsloth's Qwen3.5 fine-tuning guide loads it with `FastLanguageModel` for text
   SFT (`load_in_16bit=True`) and the usual 7 projection modules. That is what this
   notebook does. The vision tower still loads (≈0.5B params ≈1GB bf16) but gets no
   LoRA. Note: the 24 Gated-DeltaNet layers expose `in_proj_*`/`out_proj`, not
   `q/k/v/o_proj`, so on those layers only the MLP gets LoRA - same as the official
   recipe; the training cell prints which modules were wrapped so you can see it.
2. **bf16, not 4-bit.** Unsloth explicitly advises against QLoRA on Qwen3.5
   ("higher than normal quantization differences", dense and MoE alike). For a
   numeric-regression target that error lands straight on the predicted values.
3. **transformers v5 required**, and an Unsloth that knows `qwen3_5`. The pinned stack is unsloth 2026.7.5 / transformers 5.5.0;
   the install cell says what to do if that pin turns out not to load it.
4. **~40GB is not enough.** Weights alone are 18GB (vocab is 248,320 and the
   embeddings are untied, so embed+lm_head is ≈2B params by itself). **Run the
   `PROBE` cell first** — set `PROBE = True`, run once, read the peak. Do not
   start a multi-hour run on an unverified fit.

Architecture note: 24 of 32 layers are Gated DeltaNet (linear attention), only 8
are full attention, and the native window is 262k — so 32k costs far less memory
here than the layer count suggests. The real risk is not attention, it is whether
Unsloth's *vision* path keeps the same offloaded gradient checkpointing and fused
cross-entropy as the language path. That is exactly what `PROBE` measures.

Runtime → Change runtime type → **A100 GPU** (80GB).


In [ ]:
!pip install -q "unsloth==2026.7.5" "unsloth_zoo==2026.7.6" "transformers==5.5.0" "trl==0.24.0" lm-format-enforcer
import transformers, trl, unsloth
print("transformers", transformers.__version__, "| trl", trl.__version__, "| unsloth", unsloth.__version__)
assert int(transformers.__version__.split(".")[0]) >= 5, \
    "Qwen3.5 requires transformers v5; older versions will not load qwen3_5"


In [ ]:
# ── Config (mounts Drive) ────────────────────────────
# To change the learning rate (or epochs / LoRA rank), edit the values here
# and re-run this cell + the training cell. TAG / paths update automatically.
import os
# less fragmentation on a 32k-token context (recommended by the CUDA OOM message)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = "/content/drive/MyDrive/finetune"   # the uploaded finetune folder
DATA_DIR = f"{DRIVE_ROOT}/data"
SFX = ""                 # "" = with neighbors; "_nonb" = no-neighbor ablation
LR = 5e-4
EPOCHS = 3               # ~4.7k samples / eff. batch 16 = ~297 steps/epoch, so 3 ep = ~890 steps.
BASE_MODEL = "Qwen/Qwen3.5-9B"   # official Qwen weights (unified VLM checkpoint)
LOAD_4BIT = False        # Unsloth advises against QLoRA on Qwen3.5 (quantization error is
                         # larger than usual). bf16 LoRA is the supported path here.
LORA_R = 16
TRAIN_SEED = 42          # training seed
BASE_SHORT = BASE_MODEL.split("/")[-1].replace("-unsloth-bnb-4bit", "").replace("-bnb-4bit", "").lower()
TAG = (f"{BASE_SHORT}_lr{LR}_r{LORA_R}{SFX}" + ("_4bit" if LOAD_4BIT else "_bf16")
       + (f"_seed{TRAIN_SEED}" if TRAIN_SEED != 42 else ""))
OUTPUT_DIR = f"/content/iam_lora_{TAG}"          # train on fast local disk
DRIVE_CKPT = f"{DRIVE_ROOT}/checkpoints/{TAG}"   # mid-training checkpoints mirrored here
RESUME = False
PROBE = False            # True = smoke test: the 8 LONGEST train samples, 5 steps,
                         # then print peak VRAM and stop. Run this ONCE on a new model
                         # or a new precision before committing to a multi-hour run.
MAX_SEQ_LEN = 32768      # the longest training sample is 18.3k tokens. Qwen3.5 is native 262k, so this is a
                         # VRAM choice, not an architectural limit - raise it if you have room.
BATCH_SIZE = 1
SAVE_STEPS = 150         # checkpoint every 150 steps, together with each eval
EVAL_STEPS = 150         # eval on the FULL val set; SAVE_STEPS must stay a multiple of this
                         # (load_best_model_at_end requires save_steps % eval_steps == 0)
PATIENCE = 3             # early stop after 3 evals w/o improvement = 450 steps
GRAD_ACCUM = 16 // BATCH_SIZE

assert os.path.exists(f"{DATA_DIR}/sft_train{SFX}.jsonl"), f"data not found under {DATA_DIR}"
# ── split-version guard ──────────────────────────────────────────────────────
# The split under make_datasets/output/split/ is regenerated in place (ids are
# reassigned), and finetune/data must be rebuilt with prepare_sft_data.py
# afterwards. Pin the split the data were built from.
EXPECT_SPLIT = {"train": 4747, "val": 663}
import subprocess
_n = {k: int(subprocess.check_output(["wc", "-l", f"{DATA_DIR}/sft_{k}{SFX}.jsonl"]).split()[0])
      for k in EXPECT_SPLIT}
assert _n == EXPECT_SPLIT, (f"sft data is {_n}, expected {EXPECT_SPLIT} - finetune/data was built "
                            f"from an older split; re-run prepare_sft_data.py and re-upload data/")
print(TAG, "| data:", DATA_DIR, "| out:", OUTPUT_DIR)

In [ ]:
# ── Training (bf16 LoRA; loss on assistant tokens only; best checkpoint by eval_loss) ──
from unsloth import FastLanguageModel  # Unsloth's Qwen3.5 guide uses FastLanguageModel for
                                       # text SFT even though the checkpoint is a VLM
from unsloth.chat_templates import train_on_responses_only
from datasets import load_dataset
from transformers import EarlyStoppingCallback
from trl import SFTTrainer, SFTConfig

model, tokenizer = FastLanguageModel.from_pretrained(
    BASE_MODEL, max_seq_length=MAX_SEQ_LEN, load_in_4bit=LOAD_4BIT,
    load_in_16bit=not LOAD_4BIT,   # explicit bf16 LoRA (the documented Qwen3.5 path)
    dtype=None)   # dtype=None -> bf16 on A100

# Some Unsloth builds hand back a Processor for VLM checkpoints. Everything below
# is text-only, so unwrap it once and use `tok` for tokenization/length checks.
tok = getattr(tokenizer, "tokenizer", tokenizer)

model = FastLanguageModel.get_peft_model(
    model, r=LORA_R, lora_alpha=2 * LORA_R, lora_dropout=0.0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth")

# ── LoRA placement report ───────────────────────────────────────────────────
# Qwen3.5 mixes 8 full-attention layers (q/k/v/o_proj) with 24 Gated-DeltaNet
# layers (in_proj_*/out_proj). Show what actually got wrapped so the trainable
# footprint is a known quantity, and fail loudly if the vision tower got LoRA.
_lora = [n for n, _ in model.named_modules() if n.endswith("lora_A")]
_by = {}
for n in _lora:
    _by[n.split(".")[-2]] = _by.get(n.split(".")[-2], 0) + 1
print("LoRA modules by name:", _by, "| total", len(_lora))
assert not any("visual" in n or "vision" in n for n in _lora), "LoRA landed on the vision tower"
model.print_trainable_parameters()

ds = load_dataset("json", data_files={
    "train": f"{DATA_DIR}/sft_train{SFX}.jsonl",
    "val": f"{DATA_DIR}/sft_val{SFX}.jsonl"})

def to_text(ex):
    return {"text": tokenizer.apply_chat_template(
        ex["messages"], tokenize=False, add_generation_prompt=False,
        enable_thinking=False)}


# ── chat-template guard ──────────────────────────────────────────────────────
# train_on_responses_only() below masks the loss by matching literal marker
# strings. If Qwen3.5's template ever stops emitting ChatML markers, that match
# silently finds nothing and the run trains on the WHOLE sequence - a wrong
# result, not a crash. Fail here instead.
_probe_msgs = [{"role": "user", "content": "ping"},
               {"role": "assistant", "content": "pong"}]
_r = tokenizer.apply_chat_template(_probe_msgs, tokenize=False,
                                   add_generation_prompt=False,
                                   enable_thinking=False)
for _m in ("<|im_start|>user\n", "<|im_start|>assistant\n"):
    assert _m in _r, f"chat template no longer emits {_m!r}: {_r!r}"
assert "<think>" not in _r or "<think>\n\n</think>" in _r, \
    f"enable_thinking=False did not collapse the think block: {_r!r}"
print("chat template OK:", repr(_r[-120:]))

ds = ds.map(to_text, remove_columns=[c for c in ds["train"].column_names if c != "text"])

# token-length health check BEFORE training: everything must fit in MAX_SEQ_LEN,
# otherwise the assistant marker gets truncated and the sample is dropped
import numpy as np
lens = [len(x) for x in tok(list(ds["train"]["text"]))["input_ids"]]
print(f"train token lengths: p50={np.percentile(lens,50):.0f} "
      f"p95={np.percentile(lens,95):.0f} p99={np.percentile(lens,99):.0f} "
      f"max={max(lens)} | over MAX_SEQ_LEN({MAX_SEQ_LEN}): "
      f"{sum(l > MAX_SEQ_LEN for l in lens)}")
assert sum(l > MAX_SEQ_LEN for l in lens) == 0, \
    "some samples exceed MAX_SEQ_LEN - raise it in the Config cell"
vlens = [len(x) for x in tok(list(ds["val"]["text"]))["input_ids"]]
print(f"val   token lengths: p50={np.percentile(vlens,50):.0f} "
      f"p95={np.percentile(vlens,95):.0f} max={max(vlens)} | "
      f"over MAX_SEQ_LEN({MAX_SEQ_LEN}): {sum(l > MAX_SEQ_LEN for l in vlens)}")
assert sum(l > MAX_SEQ_LEN for l in vlens) == 0, \
    "some val samples exceed MAX_SEQ_LEN - raise it in the Config cell"

# ── version-safe SFTConfig kwargs ─────────────────────────────────────────────
# TRL/transformers keep renaming these; resolve them against the installed
# signature instead of hard-coding, so a version bump can never TypeError here.
#   max_seq_length -> max_length                                       (TRL >= 0.20)
#   group_by_length=True -> train_sampling_strategy="group_by_length"  (transformers 5)
import inspect
_sft_params = inspect.signature(SFTConfig.__init__).parameters
_cfg = {"max_length" if "max_length" in _sft_params else "max_seq_length": MAX_SEQ_LEN}
if BATCH_SIZE > 1:      # length grouping only saves padding when a batch holds >1 sample
    if "train_sampling_strategy" in _sft_params:
        _cfg["train_sampling_strategy"] = "group_by_length"
    elif "group_by_length" in _sft_params:
        _cfg["group_by_length"] = True

# ── survive the 24h runtime limit: mirror every checkpoint to Drive ──
# OUTPUT_DIR is local disk (fast, but wiped when the runtime dies). After each
# save (every 150 steps ≈ every eval) this callback copies the checkpoint dirs
# to DRIVE_CKPT, so a cutoff loses at most 150 steps. Resume with RESUME=True.
import glob, shutil
from transformers import TrainerCallback

class DriveCheckpointSync(TrainerCallback):
    def on_save(self, args, state, control, **kwargs):
        os.makedirs(DRIVE_CKPT, exist_ok=True)
        for p in glob.glob(f"{DRIVE_CKPT}/*.tmp"):          # leftover from a mid-copy cutoff
            shutil.rmtree(p, ignore_errors=True)
        local = {os.path.basename(p) for p in glob.glob(f"{args.output_dir}/checkpoint-*")}
        for name in sorted(local):                          # copy new ones (tmp+rename = atomic)
            dst = f"{DRIVE_CKPT}/{name}"
            if not os.path.exists(dst):
                shutil.copytree(f"{args.output_dir}/{name}", dst + ".tmp")
                os.rename(dst + ".tmp", dst)
        # keep EVERY checkpoint's adapter weights on Drive (so any step can be
        # scored on val_infer later), but only the LATEST keeps optimizer /
        # scheduler / rng state (that is what RESUME needs; ~400MB per ckpt).
        cks = sorted(glob.glob(f"{DRIVE_CKPT}/checkpoint-*"),
                     key=lambda p: int(p.rsplit("-", 1)[1]))
        for p in cks[:-1]:
            for f in glob.glob(f"{p}/optimizer.pt") + glob.glob(f"{p}/scheduler.pt") \
                     + glob.glob(f"{p}/rng_state*.pth"):
                os.remove(f)
        return control

_cfg.update(dict(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,   # effective batch 16 either way
        num_train_epochs=EPOCHS,
        learning_rate=LR,
        lr_scheduler_type="cosine",
        warmup_ratio=0.03,
        logging_steps=20,
        per_device_eval_batch_size=1,       # default is 8! at 10-21k tokens the eval
                                            # logits ([8, seq, 152k vocab] upcast to fp32)
                                            # alone are ~47GB -> OOM. eval needs no grads,
                                            # so batch 1 is the only sane setting here
        prediction_loss_only=True,          # we only read eval_loss: never keep/gather logits
        eval_strategy="steps", eval_steps=EVAL_STEPS,
        save_strategy="steps", save_steps=SAVE_STEPS,
        eval_on_start=True,                 # one eval BEFORE step 1: any eval-time
                                            # problem shows up in ~20 min instead of
                                            # 3.5h in. Set False once a run is clean.
        save_total_limit=None,              # keep every checkpoint (see DriveCheckpointSync)
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        bf16=True,
        dataset_text_field="text",
        report_to="none",
        seed=TRAIN_SEED,
))
# ── PROBE mode: 8 longest samples, 5 steps, report peak VRAM, stop ───────────
# Uses the REAL config (same MAX_SEQ_LEN, same precision, same LoRA target
# modules) on the worst-case sequences, so "it fits" here means it fits.
if PROBE:
    _longest = [int(i) for i in np.argsort(lens)[::-1][:8]]
    ds["train"] = ds["train"].select(_longest)
    ds["val"] = ds["val"].select(range(2))
    _cfg.update(max_steps=5, num_train_epochs=1, eval_steps=5, save_steps=5,
                eval_on_start=False, save_total_limit=1,
                load_best_model_at_end=False)
    print(f"[PROBE] 8 longest train samples: "
          f"{[lens[i] for i in _longest]} tokens")

_drop = [k for k in _cfg if k not in _sft_params]
if _drop:   # never crash on a renamed/removed arg - report and continue
    print("[warn] SFTConfig does not accept", _drop, "- dropping")
    for k in _drop:
        _cfg.pop(k)

# TRL renamed tokenizer= -> processing_class= (Unsloth patches the old name in
# some builds, not all); pick whichever the installed signature has.
_tk_kw = ("processing_class" if "processing_class" in inspect.signature(SFTTrainer.__init__).parameters
          else "tokenizer")
trainer = SFTTrainer(
    model=model, **{_tk_kw: tok},
    train_dataset=ds["train"], eval_dataset=ds["val"],
    callbacks=([] if PROBE else
               [EarlyStoppingCallback(early_stopping_patience=PATIENCE),
                DriveCheckpointSync()]),
    args=SFTConfig(**_cfg),
)

trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n")

import json
resume_ckpt = None
if RESUME:
    cks = sorted(glob.glob(f"{DRIVE_CKPT}/checkpoint-*"),
                 key=lambda p: int(p.rsplit("-", 1)[1]))
    assert cks, f"RESUME=True but no checkpoints under {DRIVE_CKPT}"
    for p in cks:                       # bring them back to fast local disk
        dst = f"{OUTPUT_DIR}/{os.path.basename(p)}"
        if not os.path.exists(dst):
            shutil.copytree(p, dst)
    resume_ckpt = f"{OUTPUT_DIR}/{os.path.basename(cks[-1])}"
    print("resuming from", resume_ckpt)
trainer.train(resume_from_checkpoint=resume_ckpt)
if PROBE:
    import torch
    print(f"[PROBE] PEAK VRAM: {torch.cuda.max_memory_allocated() / 2**30:.1f} GiB"
          f" / {torch.cuda.get_device_properties(0).total_memory / 2**30:.0f} GiB")
    raise SystemExit("probe finished - set PROBE=False for the real run")
# eval_loss per checkpoint - the table you pick from (best-by-eval_loss is
# picked automatically below; the others are kept for scoring on val_infer)
print("step   eval_loss")
for _e in trainer.state.log_history:
    if "eval_loss" in _e:
        print(f"{int(_e['step']):>5}  {_e['eval_loss']:.6f}")
model.save_pretrained(OUTPUT_DIR + "/best")
tokenizer.save_pretrained(OUTPUT_DIR + "/best")
# persist the train/eval-loss history alongside the adapter (survives the
# Drive backup; checkpoint dirs with trainer_state.json are NOT backed up)
with open(OUTPUT_DIR + "/best/train_log.json", "w") as _f:
    json.dump({"log_history": trainer.state.log_history,
               "best_metric": trainer.state.best_metric,
               "best_model_checkpoint": trainer.state.best_model_checkpoint},
              _f, indent=2)
_bm = trainer.state.best_metric      # None only if no eval ever ran
print(f"[{TAG}] best val loss: "
      + (f"{_bm:.4f}" if _bm is not None else "n/a")
      + f" (checkpoint {trainer.state.best_model_checkpoint})")
print("adapter saved to", OUTPUT_DIR + "/best")
import torch
print(f"peak VRAM: {torch.cuda.max_memory_allocated() / 2**30:.1f} GiB / "
      f"{torch.cuda.get_device_properties(0).total_memory / 2**30:.0f} GiB")

In [ ]:
# ── Back up the adapter (+ train log) into the Drive finetune folder ──
import shutil, os, glob
DRIVE_ADAPTER = f"{DRIVE_ROOT}/adapters/{TAG}"
os.makedirs(os.path.dirname(DRIVE_ADAPTER), exist_ok=True)
shutil.copytree(OUTPUT_DIR + "/best", DRIVE_ADAPTER, dirs_exist_ok=True)
print("adapter backed up to", DRIVE_ADAPTER)
# the per-step checkpoints stay under DRIVE_CKPT (adapter weights for every
# step; optimizer state only on the last one) - score any of them with SFT_ADAPTER in colab_infer_vllm.ipynb
print("kept per-step checkpoints:",
      sorted(os.path.basename(p) for p in glob.glob(f"{DRIVE_CKPT}/checkpoint-*")))

### If the runtime dies (24h limit / disconnect)
Two situations:

1. **Training was cut off mid-run** — checkpoints are mirrored to
   `MyDrive/finetune/checkpoints/<TAG>/` every 150 steps, so at most 150 steps are
   lost. In a fresh session: run the install + Config cells with the **same LR /
   SFX / seed** (same TAG) and set `RESUME = True`, then run the training cell —
   it continues from the last Drive checkpoint (optimizer/scheduler state included).
2. **Training already finished** (adapter backed up by the cell above) — skip
   training entirely: run install + Config, then run
   `colab_infer_vllm.ipynb`, which reads the adapter from `adapters/<TAG>`.


## Inference → `colab_infer_vllm.ipynb`
Inference runs with vLLM: open `colab_infer_vllm.ipynb` in a **fresh
runtime**, set `SFT_BASE_MODEL` / `SFT_LR` to this run's values (same TAG), pick
`SFT_SPLIT`, run. It merges the adapter under `adapters/<TAG>` (or any
`checkpoints/<TAG>/checkpoint-N`) into bf16 weights and serves it with vLLM;
output CSV lands in `MyDrive/finetune/preds/` with a `_vllm` marker.


## Evaluate back on your machine (from the project root)
Sync the CSV from `MyDrive/finetune/preds/`, then either the quick table
```bash
python3 finetune/score_preds.py --gt make_datasets/output/split/AR6_R10sd_val10_gt.xlsx \
    --pred <csv1> <csv2> ... --label 2e-4 5e-4 1e-3
```
or the full rag_core report (same multi-sheet xlsx as the RAG runs; lands next to the CSV as `<name>_eval.xlsx`):
```bash
IAM_LLM_PRED_PATH=<csv> \
IAM_LLM_BATCH_SHEET_PATH=make_datasets/output/split/AR6_R10sd_val10_list.xlsx \
IAM_LLM_GT_PATH=make_datasets/output/split/AR6_R10sd_val10_gt.xlsx \
python3 -c "import sys; sys.path.insert(0, '.'); import rag_core; rag_core.run_evaluation()"
```
(test / sub01..05: swap the list + gt paths.)
